<a href="https://colab.research.google.com/github/JorgeRojas720s/AI_Agents/blob/main/ModelBasedAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Field

In [34]:
import random
import time
import matplotlib.pyplot as plt
import numpy as np


plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

class Field:
    """
    A 4x4 Field that reṕresents where the dron should fly.
    The dron only sees its current position.
    N = Normal plant
    C = Contaminated plant
    D = Dried land
    W = Weet land
    """

    def __init__(self, seed=42):
        random.seed(seed)
        options = ["N", "C", "D", "W"]
        self.real_field = [[random.choice(options) for _ in range(4)] for _ in range(4)]

        self.position = (0, 0)
        self.log = []
        self.steps = 0

        print("Initial Field:")
        self.print_field()

    def print_field(self):
        for row in self.real_field:
            print(" ".join(row))
        print()

    def perception(self):
        x, y = self.position
        return {
            'position': self.position,
            'indicator': self.real_field[x][y]
        }

    def apply_action(self, action):
        """
        actions:
        - "up": move up
        - "down": move down
        - "left": move left
        - "right": move right
        - "water": water the plant
        - "spray": spray the plant

        :param self: self
        :param action: action to apply
        """
        self.steps += 1
        x, y = self.position

        if action == "up" and x > 0:
            self.position = (x - 1, y)
            result = "Moved up"
        elif action == "down" and x < 3:
            self.position = (x + 1, y)
            result = "Moved down"
        elif action == "left" and y > 0:
            self.position = (x, y - 1)
            result = "Moved left"
        elif action == "right" and y < 3:
            self.position = (x, y + 1)
            result = "Moved right"
        elif action == "water":
            result = "Watered the plant"
            self.real_field[x][y] = "W"
        elif action == "spray":
            result = "Sprayed the plant"
            self.real_field[x][y] = "N"
        else:
            result = "Invalid action"

        self.log.append((self.steps, self.position, action, result))
        return result

    def field_clean(self):
        for row in self.real_field:
            for cell in row:
                if cell in ["C", "D"]:
                    return False
        return True

    def treated_plants(self):
        count = 0
        for row in self.real_field:
            for cell in row:
                if cell in ["W", "N"]:
                    count += 1
        return count

if __name__ == "__main__":
    field = Field()

Initial Field:
N N D C
C C N N
W N N N
C C N C



#Agente Basado en Modelo

In [37]:
class ModelBasedAgent:

    FIELDS = [
        ['A', 'B', 'C', 'D'],
        ['E', 'F', 'G', 'H'],
        ['I', 'J', 'K', 'L'],
        ['M', 'N', 'O', 'P']
    ]

    def __init__(self):
        self.world_model = {
            cell: 'unknown'
            for row in self.FIELDS
            for cell in row
        }
        self.current_position = None

        # Maps tuple (row, col) -> cell letter
        self.cell_index = {
            (r, c): cell
            for r, row in enumerate(self.FIELDS)
            for c, cell in enumerate(row)
        }

        print('Model-based drone agent created.')

    def update_model(self, perception):
        pos = self.cell_index[perception['position']]
        self.current_position = pos
        self.world_model[pos] = perception['indicator']

    def decide_action(self, perception):
        self.update_model(perception)

        row, col = perception['position']
        pos = self.cell_index[(row, col)]
        indicator = perception['indicator']

        if indicator == 'C':
            self.world_model[pos] = 'sprayed'
            return 'spray'

        if indicator == 'D':
            self.world_model[pos] = 'watered'
            return 'water'

        attended = ('sprayed', 'watered', 'N', 'W')

        #Busca a la derecha
        for c in range(col + 1, 4):
            cell = self.cell_index[(row, c)]
            if self.world_model[cell] not in attended:
                return 'right'

        #Busca a la izquierda
        for c in range(col - 1, -1, -1):
            cell = self.cell_index[(row, c)]
            if self.world_model[cell] not in attended:
                return 'left'

        #Busca para abajo
        for r in range(row + 1, 4):
            cell = self.cell_index[(r, col)]
            if self.world_model[cell] not in attended:
                return 'down'
        #Busca para arriba
        for r in range(row - 1, -1, -1):
            cell = self.cell_index[(r, col)]
            if self.world_model[cell] not in attended:
                return 'up'

        return 'wait'

    def show_model(self):
        print('  Drone internal model:')
        for r, row in enumerate(self.FIELDS):
            for c, cell in enumerate(row):
                marker = '<--' if cell == self.current_position else '   '
                print(f'    {cell}: {self.world_model[cell]:<10} {marker}', end='')
            print()
        print()


drone = ModelBasedAgent()

Model-based drone agent created.
Initial internal state: {'A': 'unknown', 'B': 'unknown', 'C': 'unknown', 'D': 'unknown', 'E': 'unknown', 'F': 'unknown', 'G': 'unknown', 'H': 'unknown', 'I': 'unknown', 'J': 'unknown', 'K': 'unknown', 'L': 'unknown', 'M': 'unknown', 'N': 'unknown', 'O': 'unknown', 'P': 'unknown'}


# Simulación


In [38]:
def simulate_model_drone(max_steps=50):
    """Simulates the model-based drone agent over the field."""

    field = Field()
    drone = ModelBasedAgent()

    print(f'\n{"="*60}')
    print(f'  Simulation - Model-Based Drone Agent')
    print(f'{"-"*60}')
    print(f'{"Step":>5} | {"Pos":>8} | {"Indicator":>10} | Action')
    print('-' * 50)

    steps_used = 0
    for step in range(max_steps):
        perception = field.perception()
        action = drone.decide_action(perception)
        field.apply_action(action)

        print(f'{step+1:>5} | {str(perception["position"]):>8} | {perception["indicator"]:>10} | {action}')
        steps_used = step + 1

        if action == 'wait':
            print(f'\n  Drone believes it finished. Verifying...')
            if field.field_clean():
                print(f'  Correct. All fields attended in {field.steps} steps.')
            else:
                print(f'  Error: the model does not reflect reality correctly.')
            break

    print(f'\nFinal state of the drone internal model:')
    drone.show_model()
    print(f'\nReal state of the field:')
    field.print_field()
    print(f'Treated plants: {field.treated_plants()} / 16')

    return field


field_simulation = simulate_model_drone()

Initial Field:
N N D C
C C N N
W N N N
C C N C

Model-based drone agent created.
Initial internal state: {'A': 'unknown', 'B': 'unknown', 'C': 'unknown', 'D': 'unknown', 'E': 'unknown', 'F': 'unknown', 'G': 'unknown', 'H': 'unknown', 'I': 'unknown', 'J': 'unknown', 'K': 'unknown', 'L': 'unknown', 'M': 'unknown', 'N': 'unknown', 'O': 'unknown', 'P': 'unknown'}

  Simulation - Model-Based Drone Agent
------------------------------------------------------------
 Step |      Pos |  Indicator | Action
--------------------------------------------------
    1 |   (0, 0) |          N | right
    2 |   (0, 1) |          N | right
    3 |   (0, 2) |          D | water
    4 |   (0, 2) |          W | right
    5 |   (0, 3) |          C | spray
    6 |   (0, 3) |          N | down
    7 |   (1, 3) |          N | left
    8 |   (1, 2) |          N | left
    9 |   (1, 1) |          C | spray
   10 |   (1, 1) |          N | left
   11 |   (1, 0) |          C | spray
   12 |   (1, 0) |          N | d